# 04 — Genre-Based Movie Recommendation System

## Objective

Develop a personalized movie recommendation system that combines **user genre preferences** with **overall movie quality** to recommend relevant movies.

## Methodology

1. Represent each movie using binary multi-hot genre vectors.
2. Construct user preference profiles using two rating-weighting methods: signed linear weighting and positive-only weighting.
3. Calculate cosine similarity between user profiles and movie genre vectors.
4. Calculate Bayesian weighted movie ratings to account for both average ratings and rating counts.
5. Combine genre similarity and movie quality using a weighted scoring function.
6. Tune the model using a validation set and select the best configuration based on Recall@10.
7. Evaluate the selected model on an independent test set using Precision@10, Recall@10, and NDCG@10.

## Experimental Setup

- **Movie catalogue:** Movies released in 2010 or later.
- **User eligibility:** At least 200 distinct eligible movies rated.
- **Data split:** 60% training, 20% validation, and 20% testing.
- **Relevance threshold:** Ratings of 3.5 or higher out of 5.
- **Recommendation cutoff:** Top 10 movies.
- **Hyperparameter tuning:** Compare user-profile weighting methods and genre-similarity versus movie-quality weights.

## Expected Outcome

Identify a configuration that balances personalized genre similarity with overall movie quality and evaluate its ability to retrieve movies that users rated positively.

In [ ]:
import os
import ast
import json
import numpy as np
import pandas as pd

from google.colab import drive
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Define project directories
PROJECT_DIR = "/content/drive/MyDrive/MovieMatch"
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "backend_artifacts")

# Define dataset paths
MOVIES_PATH = os.path.join(ARTIFACT_DIR, "movies_clean.csv")
RATINGS_PATH = os.path.join(PROJECT_DIR, "ratings.csv")
LINKS_PATH = os.path.join(ARTIFACT_DIR, "links_columns.csv")

# Verify that all required files exist
files = {
    "Cleaned movie catalogue": MOVIES_PATH,
    "Original ratings": RATINGS_PATH,
    "MovieLens-TMDB mapping": LINKS_PATH
}

for name, path in files.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")
    print(f"Path: {path}\n")

Cleaned movie catalogue: FOUND
Path: /content/drive/MyDrive/MovieMatch/backend_artifacts/movies_clean.csv

Original ratings: FOUND
Path: /content/drive/MyDrive/MovieMatch/ratings.csv

MovieLens-TMDB mapping: FOUND
Path: /content/drive/MyDrive/MovieMatch/backend_artifacts/links_columns.csv



In [ ]:
# Load the cleaned movie catalogue
movies = pd.read_csv(MOVIES_PATH)

# Load only the required columns from the original ratings dataset
ratings = pd.read_csv(
    RATINGS_PATH,
    usecols=["userId", "movieId", "rating"]
)

# Load the existing MovieLens-to-TMDB mapping
links = pd.read_csv(
    LINKS_PATH,
    usecols=["movieId", "tmdbId"]
)

# Display dataset dimensions
print("Movie catalogue shape:", movies.shape)
print("Ratings dataset shape:", ratings.shape)
print("Movie ID mapping shape:", links.shape)

print("\nMovie catalogue columns:")
print(movies.columns.tolist())

print("\nRatings columns:")
print(ratings.columns.tolist())

print("\nMapping columns:")
print(links.columns.tolist())

Movie catalogue shape: (12794, 9)
Ratings dataset shape: (26024289, 3)
Movie ID mapping shape: (12800, 2)

Movie catalogue columns:
['genres', 'overview', 'release_date', 'title', 'release_year', 'tmdbId', 'avg_rating', 'rating_count', 'overview_clean']

Ratings columns:
['userId', 'movieId', 'rating']

Mapping columns:
['movieId', 'tmdbId']


In [ ]:
# Get the MovieLens movie IDs present in our existing mapping
eligible_movie_ids = set(links["movieId"])

# Keep only ratings for movies in our post-2010 catalogue
ratings_filtered = ratings[
    ratings["movieId"].isin(eligible_movie_ids)
].copy()

print("Original ratings:", len(ratings))
print("Ratings after filtering:", len(ratings_filtered))
print("Unique users:", ratings_filtered["userId"].nunique())
print("Unique MovieLens movie IDs:", ratings_filtered["movieId"].nunique())

Original ratings: 26024289
Ratings after filtering: 1913317
Unique users: 67167
Unique MovieLens movie IDs: 12658


In [ ]:
# Map MovieLens movie IDs to TMDB movie IDs
ratings_mapped = ratings_filtered.merge(
    links,
    on="movieId",
    how="inner",
    validate="many_to_one"
)

# Verify the mapping
print("Mapped ratings:", len(ratings_mapped))
print("Unique users:", ratings_mapped["userId"].nunique())
print("Unique MovieLens movie IDs:", ratings_mapped["movieId"].nunique())
print("Unique TMDB movie IDs:", ratings_mapped["tmdbId"].nunique())

print("\nFirst five mapped ratings:")
display(ratings_mapped.head())

Mapped ratings: 1913317
Unique users: 67167
Unique MovieLens movie IDs: 12658
Unique TMDB movie IDs: 12652

First five mapped ratings:


,userId,movieId,rating,tmdbId
0,1,81834,5.0,12444
1,1,91500,2.5,70160
2,1,91542,5.0,58574
3,1,92439,5.0,64678
4,1,96821,5.0,84892


In [ ]:
# Count distinct TMDB movies rated by each user
user_movie_counts = (
    ratings_mapped.groupby("userId")["tmdbId"]
    .nunique()
    .sort_values(ascending=False)
)

# Keep users who have rated at least 200 distinct movies
eligible_users = user_movie_counts[user_movie_counts >= 200].index

# Filter ratings to retain only eligible users
ratings_eligible = ratings_mapped[
    ratings_mapped["userId"].isin(eligible_users)
].copy()

# Display summary statistics
print("Total users before filtering:", ratings_mapped["userId"].nunique())
print("Users with at least 200 distinct movies:", len(eligible_users))
print("Ratings after user filtering:", len(ratings_eligible))
print("Unique movies rated:", ratings_eligible["tmdbId"].nunique())

print("\nDistinct movies rated per eligible user:")
display(user_movie_counts.loc[eligible_users].describe())

Total users before filtering: 67167
Users with at least 200 distinct movies: 1540
Ratings after user filtering: 526565
Unique movies rated: 11690

Distinct movies rated per eligible user:


,tmdbId
count,1540.000000
mean,341.920779
std,214.697915
min,200.000000
25%,230.000000
50%,286.000000
75%,374.250000
max,5294.000000


###Preprocess Movie Genre

In [ ]:
# Inspect the genre column
print("Movie catalogue shape:", movies.shape)

print("\nFirst five genre entries:")
display(movies[["tmdbId", "title", "genres"]].head())

print("\nData type of genres column:")
print(movies["genres"].dtype)

print("\nMissing genre entries:", movies["genres"].isna().sum())

print("\nNumber of unique genre strings:", movies["genres"].nunique())

Movie catalogue shape: (12794, 9)

First five genre entries:


,tmdbId,title,genres
0,79782,Venice,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n..."
1,141210,The Sleepover,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam..."
2,143750,The Farmer's Wife,"[{'id': 18, 'name': 'Drama'}]"
3,84198,A Place at the Table,"[{'id': 99, 'name': 'Documentary'}]"
4,171982,Romance,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."



Data type of genres column:
object

Missing genre entries: 0

Number of unique genre strings: 1683


In [ ]:
# Create a separate catalogue for the genre-based model
movies_genre = movies.copy()

# Remove rows with missing or empty genre information
movies_genre = movies_genre[
    movies_genre["genres"].notna()
    & movies_genre["genres"].str.strip().ne("")
    & movies_genre["genres"].str.strip().ne("[]")
].copy()

# Reset the index
movies_genre.reset_index(drop=True, inplace=True)

print("Original catalogue size:", len(movies))
print("Movies with genre information:", len(movies_genre))
print("Movies removed:", len(movies) - len(movies_genre))

print("\nSample of retained movies:")
display(movies_genre[["tmdbId", "title", "genres"]].head())

Original catalogue size: 12794
Movies with genre information: 12262
Movies removed: 532

Sample of retained movies:


,tmdbId,title,genres
0,79782,Venice,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n..."
1,141210,The Sleepover,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam..."
2,143750,The Farmer's Wife,"[{'id': 18, 'name': 'Drama'}]"
3,84198,A Place at the Table,"[{'id': 99, 'name': 'Documentary'}]"
4,171982,Romance,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."


In [ ]:
# Convert genre strings into lists of genre names
def extract_genre_names(genre_string):
    try:
        genre_list = ast.literal_eval(genre_string)
        return [
            genre["name"]
            for genre in genre_list
            if isinstance(genre, dict) and "name" in genre
        ]
    except (ValueError, SyntaxError, TypeError):
        return []

movies_genre["genre_list"] = movies_genre["genres"].apply(extract_genre_names)

# Inspect the results
display(movies_genre[["title", "genres", "genre_list"]].head(10))

print("Total movies:", len(movies_genre))
print("Movies with empty parsed genre lists:",
      (movies_genre["genre_list"].str.len() == 0).sum())

# Find all unique genres
all_genres = sorted({
    genre
    for genre_list in movies_genre["genre_list"]
    for genre in genre_list
})

print("\nNumber of unique genres:", len(all_genres))
print("Unique genres:")
print(all_genres)

,title,genres,genre_list
0,Venice,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[Drama, Romance]"
1,The Sleepover,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[Comedy, Horror]"
2,The Farmer's Wife,"[{'id': 18, 'name': 'Drama'}]",[Drama]
3,A Place at the Table,"[{'id': 99, 'name': 'Documentary'}]",[Documentary]
4,Romance,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[Romance, Drama, Comedy]"
5,Kingdom Come,"[{'id': 35, 'name': 'Comedy'}]",[Comedy]
6,Camille Claudel 1915,"[{'id': 18, 'name': 'Drama'}]",[Drama]
7,My Kingdom,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...","[Drama, Action, Romance, Foreign]"
8,Luster,"[{'id': 18, 'name': 'Drama'}]",[Drama]
9,Black Gold,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[Adventure, Drama]"


Total movies: 12262
Movies with empty parsed genre lists: 0

Number of unique genres: 20
Unique genres:
['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']


In [ ]:
# Create a mapping from each genre to a unique column index
genre_to_idx = {genre: idx for idx, genre in enumerate(all_genres)}

# Create a binary multi-hot vector for every movie
genre_matrix = np.zeros(
    (len(movies_genre), len(all_genres)),
    dtype=np.float32
)

for row_idx, genre_list in enumerate(movies_genre["genre_list"]):
    for genre in genre_list:
        genre_matrix[row_idx, genre_to_idx[genre]] = 1.0

print("Genre matrix shape:", genre_matrix.shape)
print("Number of genre features:", len(all_genres))

# Inspect a few movie vectors
sample_indices = [0, 1, 2]

for idx in sample_indices:
    print("\nMovie:", movies_genre.iloc[idx]["title"])
    print("Genres:", movies_genre.iloc[idx]["genre_list"])
    print("Vector:", genre_matrix[idx])
    print("Number of assigned genres:", int(genre_matrix[idx].sum()))

Genre matrix shape: (12262, 20)
Number of genre features: 20

Movie: Venice
Genres: ['Drama', 'Romance']
Vector: [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
Number of assigned genres: 2

Movie: The Sleepover
Genres: ['Comedy', 'Horror']
Vector: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
Number of assigned genres: 2

Movie: The Farmer's Wife
Genres: ['Drama']
Vector: [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Number of assigned genres: 1


In [ ]:
# Create the set of movie IDs represented in the genre catalogue
genre_movie_ids = set(movies_genre["tmdbId"])

# Check how many eligible ratings refer to movies in this catalogue
ratings_in_genre_catalogue = ratings_eligible[
    ratings_eligible["tmdbId"].isin(genre_movie_ids)
].copy()

# Check for duplicate user-movie pairs
duplicate_pairs = ratings_in_genre_catalogue.duplicated(
    subset=["userId", "tmdbId"]
).sum()

print("Eligible ratings before genre-catalogue filtering:",
      len(ratings_eligible))

print("Ratings matching the genre catalogue:",
      len(ratings_in_genre_catalogue))

print("Ratings excluded because the movie has no usable genres:",
      len(ratings_eligible) - len(ratings_in_genre_catalogue))

print("Unique users:",
      ratings_in_genre_catalogue["userId"].nunique())

print("Unique movies:",
      ratings_in_genre_catalogue["tmdbId"].nunique())

print("Duplicate user-movie pairs:", duplicate_pairs)

Eligible ratings before genre-catalogue filtering: 526565
Ratings matching the genre catalogue: 525510
Ratings excluded because the movie has no usable genres: 1055
Unique users: 1540
Unique movies: 11273
Duplicate user-movie pairs: 7


In [ ]:
# Identify every duplicated user-TMDB movie pair
duplicate_rows = ratings_in_genre_catalogue[
    ratings_in_genre_catalogue.duplicated(
        subset=["userId", "tmdbId"],
        keep=False
    )
].sort_values(["userId", "tmdbId"])

# Display the original MovieLens IDs and their mapped TMDB IDs
display(
    duplicate_rows[
        ["userId", "movieId", "tmdbId", "rating"]
    ]
)

print("Number of duplicate rows involved:", len(duplicate_rows))
print(
    "Number of affected user-TMDB pairs:",
    duplicate_rows[["userId", "tmdbId"]].drop_duplicates().shape[0]
)

,userId,movieId,tmdbId,rating
327607,45811,2679,84198,3.0
328766,45811,103208,84198,3.0
327609,45811,4484,110428,5.0
329391,45811,112062,110428,4.0
330939,45811,142861,298721,5.0
332046,45811,162632,298721,3.5
1078198,152682,4484,110428,3.5
1078352,152682,112062,110428,4.0
1357868,192319,47237,77221,4.5
1358042,192319,102174,77221,3.5


Number of duplicate rows involved: 14
Number of affected user-TMDB pairs: 7


In [ ]:
# Count rows before removing duplicated user-movie pairs
rows_before = len(ratings_in_genre_catalogue)

# Identify all user-TMDB pairs that occur more than once
duplicate_pairs = ratings_in_genre_catalogue[
    ratings_in_genre_catalogue.duplicated(
        subset=["userId", "tmdbId"],
        keep=False
    )
][["userId", "tmdbId"]].drop_duplicates()

# Remove every occurrence of those duplicated pairs
duplicate_pair_index = pd.MultiIndex.from_frame(duplicate_pairs)

ratings_genre_clean = ratings_in_genre_catalogue[
    ~pd.MultiIndex.from_frame(
        ratings_in_genre_catalogue[["userId", "tmdbId"]]
    ).isin(duplicate_pair_index)
].copy()

# Verify the results
print("Rows before removal:", rows_before)
print("Duplicated user-movie pairs removed:", len(duplicate_pairs))
print("Rating rows removed:", rows_before - len(ratings_genre_clean))
print("Rows after removal:", len(ratings_genre_clean))

print(
    "Remaining duplicate user-movie pairs:",
    ratings_genre_clean.duplicated(
        subset=["userId", "tmdbId"]
    ).sum()
)

print("Unique users remaining:", ratings_genre_clean["userId"].nunique())
print("Unique movies remaining:", ratings_genre_clean["tmdbId"].nunique())

Rows before removal: 525510
Duplicated user-movie pairs removed: 7
Rating rows removed: 14
Rows after removal: 525496
Remaining duplicate user-movie pairs: 0
Unique users remaining: 1540
Unique movies remaining: 11273


In [ ]:
# Reproducibility
RANDOM_STATE = 42

# Store each user's ratings in separate lists
train_parts = []
val_parts = []
test_parts = []

# Split each user's ratings independently
for user_id, user_ratings in ratings_genre_clean.groupby("userId"):
    user_ratings = user_ratings.sample(
        frac=1,
        random_state=RANDOM_STATE
    )

    n = len(user_ratings)
    train_end = int(0.60 * n)
    val_end = int(0.80 * n)

    train_parts.append(user_ratings.iloc[:train_end])
    val_parts.append(user_ratings.iloc[train_end:val_end])
    test_parts.append(user_ratings.iloc[val_end:])

# Combine the individual user splits
train_ratings = pd.concat(train_parts).reset_index(drop=True)
val_ratings = pd.concat(val_parts).reset_index(drop=True)
test_ratings = pd.concat(test_parts).reset_index(drop=True)

# Check the resulting split
total = len(ratings_genre_clean)

print("Total ratings:", total)

for name, data in [
    ("Training", train_ratings),
    ("Validation", val_ratings),
    ("Test", test_ratings)
]:
    print(
        f"{name}: {len(data):,} ratings "
        f"({len(data) / total * 100:.2f}%)"
    )

print("\nUnique users in each split:")
print("Training:", train_ratings["userId"].nunique())
print("Validation:", val_ratings["userId"].nunique())
print("Test:", test_ratings["userId"].nunique())

Total ratings: 525496
Training: 314,698 ratings (59.89%)
Validation: 105,100 ratings (20.00%)
Test: 105,698 ratings (20.11%)

Unique users in each split:
Training: 1540
Validation: 1540
Test: 1540


In [ ]:
# Create a directory for the shared evaluation splits
SPLIT_DIR = os.path.join(PROJECT_DIR, "evaluation_splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

# Save each split as a CSV file
train_path = os.path.join(SPLIT_DIR, "train_ratings.csv")
val_path = os.path.join(SPLIT_DIR, "val_ratings.csv")
test_path = os.path.join(SPLIT_DIR, "test_ratings.csv")

train_ratings.to_csv(train_path, index=False)
val_ratings.to_csv(val_path, index=False)
test_ratings.to_csv(test_path, index=False)

print("Evaluation splits saved successfully!")

print("\nSaved files:")
print(train_path)
print(val_path)
print(test_path)

print("\nFile sizes:")
for path in [train_path, val_path, test_path]:
    print(f"{os.path.basename(path)}: {os.path.getsize(path) / (1024**2):.2f} MB")

Evaluation splits saved successfully!

Saved files:
/content/drive/MyDrive/MovieMatch/evaluation_splits/train_ratings.csv
/content/drive/MyDrive/MovieMatch/evaluation_splits/val_ratings.csv
/content/drive/MyDrive/MovieMatch/evaluation_splits/test_ratings.csv

File sizes:
train_ratings.csv: 7.08 MB
val_ratings.csv: 2.36 MB
test_ratings.csv: 2.39 MB


In [ ]:
# Map each TMDB movie ID to its row index in the genre matrix
movie_to_idx = {
    tmdb_id: idx
    for idx, tmdb_id in enumerate(movies_genre["tmdbId"])
}

# Verify the mapping
print("Movies in genre matrix:", len(movies_genre))
print("Movie IDs in lookup:", len(movie_to_idx))

# Test the lookup with one movie
sample_movie_id = movies_genre.iloc[0]["tmdbId"]
sample_idx = movie_to_idx[sample_movie_id]

print("\nSample movie ID:", sample_movie_id)
print("Mapped row index:", sample_idx)
print("Movie title:", movies_genre.iloc[sample_idx]["title"])
print("Genre vector:", genre_matrix[sample_idx])

Movies in genre matrix: 12262
Movie IDs in lookup: 12262

Sample movie ID: 79782
Mapped row index: 0
Movie title: Venice
Genre vector: [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]


In [ ]:
# Calculate movie-level statistics using training ratings only
movie_stats = (
    train_ratings
    .groupby("tmdbId")["rating"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={
        "mean": "movie_mean_rating",
        "count": "rating_count_train"
    })
)

# Calculate the overall mean of training ratings
global_mean = train_ratings["rating"].mean()

print("Overall training rating mean:", round(global_mean, 4))
print("Movies with at least one training rating:", len(movie_stats))

print("\nTraining rating-count distribution per movie:")
display(movie_stats["rating_count_train"].describe())

print("\nSample movie statistics:")
display(movie_stats.head(10))

Overall training rating mean: 3.2277
Movies with at least one training rating: 9964

Training rating-count distribution per movie:


,rating_count_train
count,9964.000000
mean,31.583501
std,89.506701
min,1.000000
25%,2.000000
50%,4.000000
75%,13.000000
max,926.000000



Sample movie statistics:


,tmdbId,movie_mean_rating,rating_count_train
0,189,3.045620,274
1,1771,3.318321,655
2,1865,3.059886,526
3,1930,3.231523,663
4,4258,1.856061,66
5,7978,2.605882,170
6,8967,3.253546,282
7,9364,2.578947,19
8,9543,3.029279,444
9,10138,3.285535,795


In [ ]:
# Bayesian weighted rating parameters
# m is 75th percentile of rating count train
m = 13
C = global_mean

# Calculate Bayesian weighted rating for each movie
movie_stats["bayesian_rating"] = (
    (movie_stats["rating_count_train"] /
     (movie_stats["rating_count_train"] + m))
    * movie_stats["movie_mean_rating"]
    +
    (m /
     (movie_stats["rating_count_train"] + m))
    * C
)

# Inspect the results
print("Global mean rating (C):", round(C, 4))
print("Smoothing parameter (m):", m)

display(
    movie_stats[
        [
            "tmdbId",
            "movie_mean_rating",
            "rating_count_train",
            "bayesian_rating"
        ]
    ].head(10)
)

print("\nBayesian rating distribution:")
display(movie_stats["bayesian_rating"].describe())

Global mean rating (C): 3.2277
Smoothing parameter (m): 13


,tmdbId,movie_mean_rating,rating_count_train,bayesian_rating
0,189,3.045620,274,3.053869
1,1771,3.318321,655,3.316557
2,1865,3.059886,526,3.063934
3,1930,3.231523,663,3.231450
4,4258,1.856061,66,2.081776
5,7978,2.605882,170,2.650056
6,8967,3.253546,282,3.252408
7,9364,2.578947,19,2.842509
8,9543,3.029279,444,3.034924
9,10138,3.285535,795,3.284604



Bayesian rating distribution:


,bayesian_rating
count,9964.000000
mean,3.165451
std,0.213289
min,1.785985
25%,3.064018
50%,3.197352
75%,3.282877
max,4.068461


In [ ]:
# Identify movies that have no ratings in the training dataset
rated_movie_ids = set(movie_stats["tmdbId"])
all_genre_movie_ids = set(movies_genre["tmdbId"])

unrated_movie_ids = all_genre_movie_ids - rated_movie_ids

# Extract details of unrated movies
unrated_movies = movies_genre[
    movies_genre["tmdbId"].isin(unrated_movie_ids)
].copy()

print("Total movies in genre catalogue:", len(movies_genre))
print("Movies with training ratings:", len(rated_movie_ids))
print("Movies without training ratings:", len(unrated_movies))

print(
    "\nPercentage of catalogue without training ratings:",
    round(len(unrated_movies) / len(movies_genre) * 100, 2),
    "%"
)

print("\nSample movies without training ratings:")
display(unrated_movies[["tmdbId", "title", "genre_list"]].head(15))

Total movies in genre catalogue: 12262
Movies with training ratings: 9964
Movies without training ratings: 2298

Percentage of catalogue without training ratings: 18.74 %

Sample movies without training ratings:


,tmdbId,title,genre_list
0,79782,Venice,"[Drama, Romance]"
10,171771,The Scar,[Drama]
60,68987,Cooking With Stella,"[Drama, Comedy, Romance]"
162,139433,No One Dies in Lily Dale,[Documentary]
173,35716,Preacher's Kid,"[Drama, Family]"
271,162643,Snow & Ashes,"[Drama, Thriller, War]"
287,37690,Get Educated: Paathshaala,"[Drama, Foreign]"
321,56320,The Scientist,"[Drama, Science Fiction]"
520,45033,20 Seconds of Joy,[Documentary]
528,65289,A Nightmare in Las Cruces,"[Documentary, Drama]"


In [ ]:
# Create a quality-score table for every movie in the genre catalogue
movie_quality = movies_genre[["tmdbId", "title"]].copy()

# Add training-derived movie statistics
movie_quality = movie_quality.merge(
    movie_stats[
        ["tmdbId", "movie_mean_rating", "rating_count_train",
         "bayesian_rating"]
    ],
    on="tmdbId",
    how="left",
    validate="one_to_one"
)

# Movies without training ratings receive the global training mean
movie_quality["bayesian_rating"] = (
    movie_quality["bayesian_rating"].fillna(global_mean)
)

# Their missing training statistics remain missing for transparency
print("Total movies:", len(movie_quality))
print(
    "Movies without training ratings:",
    movie_quality["rating_count_train"].isna().sum()
)
print(
    "Movies assigned global mean:",
    (movie_quality["rating_count_train"].isna()).sum()
)

print("\nQuality-score examples:")
display(
    movie_quality[
        ["tmdbId", "title", "rating_count_train", "bayesian_rating"]
    ].head(10)
)

print("\nFinal quality-score distribution:")
display(movie_quality["bayesian_rating"].describe())

Total movies: 12262
Movies without training ratings: 2298
Movies assigned global mean: 2298

Quality-score examples:


,tmdbId,title,rating_count_train,bayesian_rating
0,79782,Venice,NaN,3.227714
1,141210,The Sleepover,2.0,3.130685
2,143750,The Farmer's Wife,8.0,2.879061
3,84198,A Place at the Table,8.0,3.140966
4,171982,Romance,23.0,2.735008
5,136558,Kingdom Come,6.0,2.945278
6,110428,Camille Claudel 1915,14.0,3.480010
7,78022,My Kingdom,1.0,3.175734
8,66111,Luster,4.0,3.115310
9,77221,Black Gold,6.0,3.366330



Final quality-score distribution:


,bayesian_rating
count,12262.000000
mean,3.177119
std,0.193794
min,1.785985
25%,3.097352
50%,3.227714
75%,3.248581
max,4.068461


In [ ]:
# Build user genre profiles using signed linear weighting
# Weighting rule: w(r) = r - 3

user_profiles_signed = {}

for user_id, user_data in train_ratings.groupby("userId"):

    # Initialize a 20-dimensional profile for this user
    profile = np.zeros(len(all_genres), dtype=np.float32)

    for row in user_data.itertuples(index=False):

        tmdb_id = row.tmdbId
        rating = row.rating

        # Skip movies missing from the genre catalogue
        if tmdb_id not in movie_to_idx:
            continue

        # Retrieve the movie's genre vector
        movie_vector = genre_matrix[movie_to_idx[tmdb_id]]

        # Calculate signed rating weight
        weight = rating - 3.0

        # Add the weighted genre vector to the user's profile
        profile += weight * movie_vector

    user_profiles_signed[user_id] = profile

# Inspect the resulting profiles
print("Number of user profiles:", len(user_profiles_signed))
print("Number of genre dimensions:", len(all_genres))

sample_user_id = next(iter(user_profiles_signed))

print("\nSample user ID:", sample_user_id)
print("Profile vector:", user_profiles_signed[sample_user_id])

print("\nGenre preferences for this user:")
display(
    pd.DataFrame({
        "genre": all_genres,
        "profile_weight": user_profiles_signed[sample_user_id]
    }).sort_values("profile_weight", ascending=False)
)

Number of user profiles: 1540
Number of genre dimensions: 20

Sample user ID: 231
Profile vector: [ 5.5  1.   0.5  2.5  7.   0.   7.5  1.5 -1.5  0.   1.5 -3.5  0.5  1.5
 -0.5 -3.5  0.   7.  -1.5  0.5]

Genre preferences for this user:


,genre,profile_weight
6,Drama,7.5
4,Crime,7.0
17,Thriller,7.0
0,Action,5.5
3,Comedy,2.5
10,History,1.5
13,Mystery,1.5
7,Family,1.5
1,Adventure,1.0
2,Animation,0.5


In [ ]:
# Build user genre profiles using positive-only weighting
# Weighting rule: w(r) = max(0, r - 3)

user_profiles_positive = {}

for user_id, user_data in train_ratings.groupby("userId"):

    # Initialize a 20-dimensional profile for this user
    profile = np.zeros(len(all_genres), dtype=np.float32)

    for row in user_data.itertuples(index=False):

        tmdb_id = row.tmdbId
        rating = row.rating

        # Skip movies missing from the genre catalogue
        if tmdb_id not in movie_to_idx:
            continue

        # Retrieve the movie's genre vector
        movie_vector = genre_matrix[movie_to_idx[tmdb_id]]

        # Calculate positive-only rating weight
        weight = max(0.0, rating - 3.0)

        # Add the weighted genre vector to the user's profile
        profile += weight * movie_vector

    user_profiles_positive[user_id] = profile


# Inspect the resulting profiles
print("Number of user profiles:", len(user_profiles_positive))
print("Number of genre dimensions:", len(all_genres))

sample_user_id = next(iter(user_profiles_positive))

print("\nSample user ID:", sample_user_id)
print("Profile vector:", user_profiles_positive[sample_user_id])

print("\nGenre preferences for this user:")
display(
    pd.DataFrame({
        "genre": all_genres,
        "profile_weight": user_profiles_positive[sample_user_id]
    }).sort_values("profile_weight", ascending=False)
)

Number of user profiles: 1540
Number of genre dimensions: 20

Sample user ID: 231
Profile vector: [15.   6.   1.  13.   9.   0.  23.   2.   2.5  0.   2.5  1.   0.5  3.
  3.5  4.   0.  18.   0.   0.5]

Genre preferences for this user:


,genre,profile_weight
6,Drama,23.0
17,Thriller,18.0
0,Action,15.0
3,Comedy,13.0
4,Crime,9.0
1,Adventure,6.0
15,Science Fiction,4.0
14,Romance,3.5
13,Mystery,3.0
10,History,2.5


In [ ]:
# Calculate cosine similarity between user profiles and movie genre vectors

# Convert user profiles into matrices
user_ids = list(user_profiles_signed.keys())

signed_profile_matrix = np.vstack([
    user_profiles_signed[user_id]
    for user_id in user_ids
])

positive_profile_matrix = np.vstack([
    user_profiles_positive[user_id]
    for user_id in user_ids
])

# Normalize user profiles for cosine similarity
def normalize_profiles(profile_matrix):
    norms = np.linalg.norm(profile_matrix, axis=1, keepdims=True)

    # Avoid division by zero for users with an empty profile
    norms[norms == 0] = 1.0

    return profile_matrix / norms


signed_profile_normalized = normalize_profiles(signed_profile_matrix)
positive_profile_normalized = normalize_profiles(positive_profile_matrix)

# Normalize movie genre vectors
movie_genre_norms = np.linalg.norm(genre_matrix, axis=1, keepdims=True)
movie_genre_normalized = genre_matrix / movie_genre_norms

# Compute user-to-movie cosine similarities
signed_similarity = signed_profile_normalized @ movie_genre_normalized.T
positive_similarity = positive_profile_normalized @ movie_genre_normalized.T

print("Number of users:", len(user_ids))
print("Number of movies:", len(movies_genre))

print("\nSigned similarity matrix shape:", signed_similarity.shape)
print("Positive-only similarity matrix shape:", positive_similarity.shape)

print("\nSigned similarity range:",
      signed_similarity.min(), "to", signed_similarity.max())

print("Positive-only similarity range:",
      positive_similarity.min(), "to", positive_similarity.max())

Number of users: 1540
Number of movies: 12262

Signed similarity matrix shape: (1540, 12262)
Positive-only similarity matrix shape: (1540, 12262)

Signed similarity range: -0.9588521 to 0.9677544
Positive-only similarity range: 0.0 to 0.99999994


In [ ]:
# Normalize Bayesian movie quality scores to [0, 1]

# Use only the Bayesian scores derived from training ratings
# to establish the normalization bounds.
rated_movie_scores = movie_quality.loc[
    movie_quality["rating_count_train"].notna(),
    "bayesian_rating"
]

bayesian_min = rated_movie_scores.min()
bayesian_max = rated_movie_scores.max()

# Apply the same transformation to the entire catalogue.
# Movies without training ratings already have the training-set
# global mean assigned as their Bayesian rating.
movie_quality["quality_normalized"] = (
    (movie_quality["bayesian_rating"] - bayesian_min)
    / (bayesian_max - bayesian_min)
)

print("Bayesian rating minimum:", bayesian_min)
print("Bayesian rating maximum:", bayesian_max)

print("\nNormalized quality statistics:")
print(movie_quality["quality_normalized"].describe())

print(
    "\nMissing normalized quality scores:",
    movie_quality["quality_normalized"].isna().sum()
)

print("\nSample movies:")
display(
    movie_quality[
        ["tmdbId", "title", "bayesian_rating", "quality_normalized"]
    ].head(10)
)

Bayesian rating minimum: 1.7859852666778673
Bayesian rating maximum: 4.0684607042323355

Normalized quality statistics:
count    12262.000000
mean         0.609485
std          0.084905
min          0.000000
25%          0.574537
50%          0.631651
75%          0.640794
max          1.000000
Name: quality_normalized, dtype: float64

Missing normalized quality scores: 0

Sample movies:


,tmdbId,title,bayesian_rating,quality_normalized
0,79782,Venice,3.227714,0.631651
1,141210,The Sleepover,3.130685,0.589141
2,143750,The Farmer's Wife,2.879061,0.478899
3,84198,A Place at the Table,3.140966,0.593645
4,171982,Romance,2.735008,0.415786
5,136558,Kingdom Come,2.945278,0.507910
6,110428,Camille Claudel 1915,3.480010,0.742188
7,78022,My Kingdom,3.175734,0.608878
8,66111,Luster,3.115310,0.582405
9,77221,Black Gold,3.366330,0.692382


In [ ]:

# Normalize genre similarity scores for both profile methods

# Signed linear similarity: transform cosine similarity from [-1, 1] to [0, 1]
signed_similarity_normalized = (signed_similarity + 1.0) / 2.0

# Positive-only similarity is already in [0, 1]
positive_similarity_normalized = np.clip(
    positive_similarity, 0.0, 1.0
)

# Check the resulting ranges
print("Signed normalized similarity:")
print("Minimum:", signed_similarity_normalized.min())
print("Maximum:", signed_similarity_normalized.max())

print("\nPositive-only similarity:")
print("Minimum:", positive_similarity_normalized.min())
print("Maximum:", positive_similarity_normalized.max())

print("\nMatrix shapes:")
print("Signed:", signed_similarity_normalized.shape)
print("Positive-only:", positive_similarity_normalized.shape)
print("Bayesian quality:", movie_quality["quality_normalized"].shape)


Signed normalized similarity:
Minimum: 0.020573944
Maximum: 0.9838772

Positive-only similarity:
Minimum: 0.0
Maximum: 0.99999994

Matrix shapes:
Signed: (1540, 12262)
Positive-only: (1540, 12262)
Bayesian quality: (12262,)


In [ ]:
# Verify alignment before hyperparameter tuning

# Check that the user order matches both similarity matrices
assert signed_similarity_normalized.shape[0] == len(user_ids)
assert positive_similarity_normalized.shape[0] == len(user_ids)

# Check that the movie order matches the genre matrix
assert signed_similarity_normalized.shape[1] == len(movies_genre)
assert positive_similarity_normalized.shape[1] == len(movies_genre)
assert len(movie_quality) == len(movies_genre)

# Check that Bayesian quality scores are valid
assert movie_quality["quality_normalized"].notna().all()
assert movie_quality["quality_normalized"].between(0, 1).all()

# Check that every user in the saved training set has a profile
train_user_ids = set(train_ratings["userId"].unique())
profile_user_ids = set(user_ids)

assert train_user_ids == profile_user_ids

# Check that each movie ID maps to its correct row
assert len(movie_to_idx) == len(movies_genre)

print("All alignment checks passed!")
print("Users:", len(user_ids))
print("Movies:", len(movies_genre))
print("Configurations to evaluate:", 10)
print("Relevance threshold: rating >= 3.5")
print("Recommendation cutoff: K = 10")

All alignment checks passed!
Users: 1540
Movies: 12262
Configurations to evaluate: 10
Relevance threshold: rating >= 3.5
Recommendation cutoff: K = 10


In [ ]:
# Prepare training histories and validation relevance sets

K = 10
RELEVANCE_THRESHOLD = 3.5

# Movies already rated by each user in the training set
train_seen = (
    train_ratings.groupby("userId")["tmdbId"]
    .apply(set)
    .to_dict()
)

# Relevant validation movies: ratings >= 3.5
relevant_val = (
    val_ratings[val_ratings["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("userId")["tmdbId"]
    .apply(set)
    .to_dict()
)

# Count users who have at least one relevant validation movie
users_with_relevant_val = sum(
    len(relevant_val.get(user_id, set())) > 0
    for user_id in user_ids
)

print("Number of users:", len(user_ids))
print("Users with training history:", len(train_seen))
print("Users with at least one relevant validation movie:",
      users_with_relevant_val)

print("\nTotal relevant user-movie pairs in validation:",
      sum(len(items) for items in relevant_val.values()))

# Inspect one user
sample_user = user_ids[0]

print("\nSample user:", sample_user)
print("Movies rated in training:", len(train_seen.get(sample_user, set())))
print("Relevant validation movies:",
      len(relevant_val.get(sample_user, set())))

Number of users: 1540
Users with training history: 1540
Users with at least one relevant validation movie: 1538

Total relevant user-movie pairs in validation: 56628

Sample user: 231
Movies rated in training: 186
Relevant validation movies: 18


In [ ]:
# Evaluation metrics for one user's top-K recommendations

def evaluate_user_recommendations(recommended_movies, relevant_movies, k=10):
    """
    Evaluate one user's ranked recommendation list.

    Parameters
    ----------
    recommended_movies : list
        Movie IDs ranked from highest to lowest recommendation score.

    relevant_movies : set
        Movie IDs rated >= RELEVANCE_THRESHOLD in validation.

    k : int
        Number of recommendations to evaluate.

    Returns
    -------
    dict
        Precision@K, Recall@K, and NDCG@K.
        Recall is NaN when the user has no relevant validation movies.
    """

    # Keep the first K recommendations
    recommended_movies = recommended_movies[:k]

    # Binary relevance: 1 if the movie is relevant, otherwise 0
    relevance = np.array(
        [1 if movie_id in relevant_movies else 0
         for movie_id in recommended_movies],
        dtype=np.float32
    )

    # Precision@K
    precision = relevance.sum() / k

    # Recall@K
    if len(relevant_movies) > 0:
        recall = relevance.sum() / len(relevant_movies)
    else:
        recall = np.nan

    # NDCG@K
    # Rank 1 receives discount 1/log2(2), rank 2 receives 1/log2(3), etc.
    discounts = 1.0 / np.log2(np.arange(2, len(relevance) + 2))

    dcg = np.sum(relevance * discounts)

    # Ideal DCG: all relevant movies appear at the top
    ideal_relevance = np.ones(
        min(len(relevant_movies), k),
        dtype=np.float32
    )

    ideal_discounts = 1.0 / np.log2(
        np.arange(2, len(ideal_relevance) + 2)
    )

    idcg = np.sum(ideal_relevance * ideal_discounts)

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        f"Precision@{k}": precision,
        f"Recall@{k}": recall,
        f"NDCG@{k}": ndcg
    }


# Quick sanity check using the sample user
sample_relevant = {101, 102, 103, 104}
sample_recommendations = [101, 999, 102, 888, 777, 666, 555, 444, 333, 222]

sample_metrics = evaluate_user_recommendations(
    sample_recommendations,
    sample_relevant,
    k=10
)

print("Sample recommendations:", sample_recommendations)
print("Relevant movies:", sample_relevant)
print("Metrics:", sample_metrics)

Sample recommendations: [101, 999, 102, 888, 777, 666, 555, 444, 333, 222]
Relevant movies: {104, 101, 102, 103}
Metrics: {'Precision@10': np.float32(0.2), 'Recall@10': np.float32(0.5), 'NDCG@10': np.float64(0.5855700749881525)}


In [ ]:
# Function to generate top-K recommendations for a user

def recommend_movies(user_id, alpha, profile_method, k=10):

    # Select the appropriate normalized similarity matrix
    if profile_method == "signed":
        similarity_matrix = signed_similarity_normalized
    elif profile_method == "positive":
        similarity_matrix = positive_similarity_normalized
    else:
        raise ValueError(
            "profile_method must be 'signed' or 'positive'"
        )

    # Find the user's row in the similarity matrix
    user_idx = user_ids.index(user_id)

    # Get this user's genre similarity scores for all movies
    genre_scores = similarity_matrix[user_idx]

    # Get normalized Bayesian quality scores for all movies
    quality_scores = movie_quality["quality_normalized"].to_numpy()

    # Combine genre similarity and Bayesian quality
    combined_scores = (
        alpha * genre_scores
        + (1 - alpha) * quality_scores
    )

    # Exclude movies the user rated in the training set
    seen_movies = train_seen.get(user_id, set())

    # Identify movies not present in the user's training history
    candidate_mask = ~movies_genre["tmdbId"].isin(seen_movies).to_numpy()

    candidate_indices = np.flatnonzero(candidate_mask)

    # Rank eligible candidates by combined score
    ranked_indices = candidate_indices[
        np.argsort(combined_scores[candidate_indices])[::-1]
    ]

    # Select the top K movies
    top_indices = ranked_indices[:k]

    # Return movie IDs, titles, and score components
    recommendations = movies_genre.iloc[top_indices][
        ["tmdbId", "title"]
    ].copy()

    recommendations["genre_similarity"] = genre_scores[top_indices]
    recommendations["bayesian_quality"] = quality_scores[top_indices]
    recommendations["combined_score"] = combined_scores[top_indices]

    return recommendations.reset_index(drop=True)


# Test the function for one user and one configuration
sample_recommendations = recommend_movies(
    user_id=231,
    alpha=0.8,
    profile_method="signed",
    k=10
)

display(sample_recommendations)

,tmdbId,title,genre_similarity,bayesian_quality,combined_score
0,273481,Sicario,0.921587,0.855914,0.908452
1,242582,Nightcrawler,0.910587,0.895954,0.907660
2,146233,Prisoners,0.910587,0.874080,0.903285
3,130747,The Guardians,0.946540,0.685492,0.894331
4,158091,Metro Manila,0.946540,0.672088,0.891650
5,80188,Boy Wonder,0.946540,0.669167,0.891066
6,171846,Killers,0.946540,0.664788,0.890190
7,23168,The Town,0.910587,0.803490,0.889167
8,50348,The Lincoln Lawyer,0.910587,0.801213,0.888712
9,97340,Black's Game,0.946540,0.647557,0.886744


### Hyperparameter Tuning and Validation

We evaluate 10 configurations by combining two user-profile weighting methods (signed linear and positive-only) with five values of alpha (1.0, 0.9, 0.8, 0.7, 0.5). For each configuration, we combine genre similarity and Bayesian movie quality scores to recommend the top 10 movies to each user, excluding movies rated in the training set. We evaluate the recommendations against relevant validation movies (rating ≥ 3.5) using Precision@10, Recall@10, and NDCG@10.

**We primarily use mean Recall@10 as our selection metric** because our main objective is to recommend as many movies as possible that a user would genuinely like, among their top 10 recommendations. Precision@10 and NDCG@10 serve as supporting metrics to assess recommendation accuracy and ranking quality. We calculate the metrics for each user and average them across users to compare configurations and select the best-performing parameters based primarily on mean Recall@10.

In [ ]:
# Hyperparameter tuning on the validation set

ALPHA_VALUES = [1.0, 0.9, 0.8, 0.7, 0.5]
PROFILE_METHODS = ["signed", "positive"]
K = 10

# Map each user ID to the corresponding similarity-matrix row
user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

# Prepare movie IDs in exactly the same order as the similarity columns
catalogue_movie_ids = movies_genre["tmdbId"].to_numpy()

# Normalized Bayesian quality scores, aligned with catalogue_movie_ids
quality_scores = movie_quality["quality_normalized"].to_numpy()

# Precompute unseen candidate indices for each user.
# Only training-rated movies are excluded.
candidate_indices_by_user = {}

for user_id in user_ids:
    seen_movies = train_seen.get(user_id, set())

    candidate_mask = ~np.isin(catalogue_movie_ids, list(seen_movies))

    candidate_indices_by_user[user_id] = np.flatnonzero(candidate_mask)


# Store configuration-level and user-level results
tuning_results = []
per_user_results = []

# Evaluate every profile method and alpha value
for profile_method in PROFILE_METHODS:

    if profile_method == "signed":
        similarity_matrix = signed_similarity_normalized
    else:
        similarity_matrix = positive_similarity_normalized

    for alpha in ALPHA_VALUES:

        print(f"Evaluating method={profile_method}, alpha={alpha}")

        # Combined scores for every user-movie pair
        combined_score_matrix = (
            alpha * similarity_matrix
            + (1.0 - alpha) * quality_scores[np.newaxis, :]
        )

        # Evaluate recommendations separately for each user
        for user_id in user_ids:

            user_idx = user_to_idx[user_id]
            candidate_indices = candidate_indices_by_user[user_id]

            user_scores = combined_score_matrix[user_idx]

            # Find the top K candidates without fully sorting the catalogue
            candidate_scores = user_scores[candidate_indices]

            top_k_local = np.argpartition(
                candidate_scores, -K
            )[-K:]

            # Sort the selected top K by descending score
            top_k_local = top_k_local[
                np.argsort(candidate_scores[top_k_local])[::-1]
            ]

            top_movie_indices = candidate_indices[top_k_local]
            recommended_movie_ids = catalogue_movie_ids[top_movie_indices]

            # Relevant validation movies for this user
            relevant_movies = relevant_val.get(user_id, set())

            # Calculate Precision@10, Recall@10, and NDCG@10
            metrics = evaluate_user_recommendations(
                recommended_movies=recommended_movie_ids.tolist(),
                relevant_movies=relevant_movies,
                k=K
            )

            per_user_results.append({
                "userId": user_id,
                "profile_method": profile_method,
                "alpha": alpha,
                **metrics
            })

        # Aggregate metrics for the current configuration
        current_results = [
            row for row in per_user_results
            if row["profile_method"] == profile_method
            and row["alpha"] == alpha
        ]

        current_df = pd.DataFrame(current_results)

        tuning_results.append({
            "profile_method": profile_method,
            "alpha": alpha,
            "mean_Precision@10": current_df["Precision@10"].mean(),
            "mean_Recall@10": current_df["Recall@10"].mean(),
            "mean_NDCG@10": current_df["NDCG@10"].mean(),
            "users_evaluated": len(current_df),
            "users_in_Recall@10": current_df["Recall@10"].notna().sum()
        })


# Create summary tables
tuning_summary = pd.DataFrame(tuning_results)

per_user_metrics = pd.DataFrame(per_user_results)

# Rank configurations by mean Recall@10
tuning_summary = tuning_summary.sort_values(
    by=["mean_Recall@10", "mean_NDCG@10", "mean_Precision@10"],
    ascending=False
).reset_index(drop=True)

print("\nHyperparameter tuning completed!")

print("\nResults ranked by mean Recall@10:")
display(tuning_summary)

print("\nBest configuration:")
display(tuning_summary.head(1))

Evaluating method=signed, alpha=1.0
Evaluating method=signed, alpha=0.9
Evaluating method=signed, alpha=0.8
Evaluating method=signed, alpha=0.7
Evaluating method=signed, alpha=0.5
Evaluating method=positive, alpha=1.0
Evaluating method=positive, alpha=0.9
Evaluating method=positive, alpha=0.8
Evaluating method=positive, alpha=0.7
Evaluating method=positive, alpha=0.5

Hyperparameter tuning completed!

Results ranked by mean Recall@10:


,profile_method,alpha,mean_Precision@10,mean_Recall@10,mean_NDCG@10,users_evaluated,users_in_Recall@10
0,positive,0.5,0.166623,0.052014,0.180367,1540,1538
1,signed,0.5,0.144351,0.039736,0.157590,1540,1538
2,signed,0.7,0.125844,0.034191,0.139173,1540,1538
3,positive,0.7,0.106234,0.033476,0.121156,1540,1538
4,signed,0.8,0.097143,0.026714,0.111301,1540,1538
5,positive,0.8,0.074351,0.023162,0.084261,1540,1538
6,signed,0.9,0.064286,0.017870,0.074669,1540,1538
7,positive,0.9,0.049935,0.015908,0.054806,1540,1538
8,positive,1.0,0.017727,0.005346,0.017117,1540,1538
9,signed,1.0,0.015130,0.004116,0.014382,1540,1538



Best configuration:


,profile_method,alpha,mean_Precision@10,mean_Recall@10,mean_NDCG@10,users_evaluated,users_in_Recall@10
0,positive,0.5,0.166623,0.052014,0.180367,1540,1538


In [ ]:
# Final configuration selected using the validation set

BEST_PROFILE_METHOD = "positive"
BEST_ALPHA = 0.5
K = 10
RELEVANCE_THRESHOLD = 3.5

print("Selected genre recommender configuration")
print(f"Profile method: {BEST_PROFILE_METHOD}")
print(f"Alpha: {BEST_ALPHA}")
print(f"Genre similarity weight: {BEST_ALPHA}")
print(f"Bayesian quality weight: {1 - BEST_ALPHA}")
print(f"Top K recommendations: {K}")
print(f"Relevance threshold: {RELEVANCE_THRESHOLD}")

Selected genre recommender configuration
Profile method: positive
Alpha: 0.5
Genre similarity weight: 0.5
Bayesian quality weight: 0.5
Top K recommendations: 10
Relevance threshold: 3.5


In [ ]:

# Prepare relevant movies from the test set

relevant_test = (
    test_ratings[test_ratings["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("userId")["tmdbId"]
    .apply(set)
    .to_dict()
)

print("Number of users in test set:", test_ratings["userId"].nunique())
print("Users with at least one relevant test movie:", len(relevant_test))

total_relevant_test = sum(
    len(movie_set) for movie_set in relevant_test.values()
)

print("Total relevant user-movie pairs:", total_relevant_test)


Number of users in test set: 1540
Users with at least one relevant test movie: 1537
Total relevant user-movie pairs: 57247


In [ ]:

# Final test-set evaluation

test_results = []

# Use the selected positive-only similarity matrix
similarity_matrix = positive_similarity_normalized

# Movie quality scores
quality_scores = movie_quality["quality_normalized"].to_numpy()

# Combined scores using the selected alpha
combined_score_matrix = (
    BEST_ALPHA * similarity_matrix
    + (1 - BEST_ALPHA) * quality_scores[np.newaxis, :]
)

catalogue_movie_ids = movies_genre["tmdbId"].to_numpy()

for user_id in user_ids:

    user_idx = user_to_idx[user_id]

    # Get candidate movies (exclude training-rated movies)
    candidate_indices = candidate_indices_by_user[user_id]

    # Calculate scores for candidate movies
    candidate_scores = combined_score_matrix[user_idx, candidate_indices]

    # Select top 10 movies
    top_k_local = np.argpartition(candidate_scores, -K)[-K:]

    top_k_local = top_k_local[
        np.argsort(candidate_scores[top_k_local])[::-1]
    ]

    recommended_indices = candidate_indices[top_k_local]

    recommended_movie_ids = catalogue_movie_ids[recommended_indices]

    # Evaluate against relevant test movies
    relevant_movies = relevant_test.get(user_id, set())

    metrics = evaluate_user_recommendations(
        recommended_movies=recommended_movie_ids.tolist(),
        relevant_movies=relevant_movies,
        k=K
    )

    test_results.append({
        "userId": user_id,
        **metrics
    })

# Convert results into a DataFrame
test_metrics_df = pd.DataFrame(test_results)

print("Test-set evaluation completed!")
print("Number of evaluated users:", len(test_metrics_df))

display(test_metrics_df.head())


Test-set evaluation completed!
Number of evaluated users: 1540


,userId,Precision@10,Recall@10,NDCG@10
0,231,0.0,0.000000,0.000000
1,719,0.4,0.129032,0.527106
2,741,0.4,0.065574,0.456997
3,836,0.1,0.018868,0.078398
4,930,0.2,0.068966,0.305235


In [ ]:

# Calculate macro-averaged test metrics

final_test_metrics = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Recall@10",
        "NDCG@10"
    ],
    "Score": [
        test_metrics_df["Precision@10"].mean(),
        test_metrics_df["Recall@10"].mean(),
        test_metrics_df["NDCG@10"].mean()
    ]
})

print("FINAL GENRE RECOMMENDER TEST RESULTS")
print("------------------------------------")

display(final_test_metrics)

print("Total users evaluated:", len(test_metrics_df))
print(
    "Users included in Recall calculation:",
    test_metrics_df["Recall@10"].notna().sum()
)


FINAL GENRE RECOMMENDER TEST RESULTS
------------------------------------


,Metric,Score
0,Precision@10,0.169805
1,Recall@10,0.052724
2,NDCG@10,0.183582


Total users evaluated: 1540
Users included in Recall calculation: 1537


### Conclusion and Analysis of Genre-Based Recommendation System

The genre-based recommendation system achieved the following results on the test set using the selected configuration (Positive-only user profiles, α = 0.5):

| Evaluation Metric | Score |
|---|---:|
| Precision@10 | 0.1698 |
| Recall@10 | 0.0527 |
| NDCG@10 | 0.1836 |

The model combines user genre preferences with Bayesian movie quality, giving equal importance to both components.

**Reasons for the relatively low evaluation scores:**

1. **Incomplete user feedback:** Our evaluation considers a recommended movie relevant only if the user has watched it and rated it at least 3.5 in the test set. However, a recommended movie might genuinely appeal to the user even if they have never watched or rated it. Such recommendations are counted as irrelevant during evaluation, even though they could potentially be good recommendations.

2. **Limited recommendation list:** We recommend only the top 10 movies to each user, whereas users may have hundreds of movies they would enjoy. Consequently, the model can retrieve only a small fraction of all relevant movies, contributing to the relatively low Recall@10.

3. **Limited genre representation:** Movies are represented using binary genre vectors. This captures broad genre preferences but cannot distinguish between movies within the same genre based on their storyline, themes, actors, directors, or other characteristics. Therefore, two movies belonging to similar genres may receive comparable scores despite appealing to different audiences.

4. **Equal weighting of genre similarity and movie quality:** The selected configuration assigns 50% weight to genre similarity and 50% to Bayesian movie quality. Although this combination performed best among our tested configurations, movie quality does not necessarily reflect an individual user's preferences. A generally well-rated movie may not be particularly suitable for a specific user.

5. **User preference complexity:** The positive-only profile considers movies rated above 3.0 as positive feedback and ignores movies rated 3.0 or below. This simplifies preference modelling but loses some information about movies a user dislikes or feels neutral about.

**Overall conclusion:**

Despite these limitations, the genre-based recommender provides a simple and interpretable approach to personalized movie recommendations. It combines individual genre preferences with the overall quality of movies to produce recommendations. The results also demonstrate that incorporating movie quality alongside genre similarity improves validation performance compared with relying exclusively on genre similarity in our tested configurations.

In [ ]:

# Prepare genre model artifacts

catalogue_movie_ids = movies_genre["tmdbId"].to_numpy()
catalogue_titles = movies_genre["title"].to_numpy()

# Verify that movie quality and genre catalogue have identical ordering
assert np.array_equal(
    movie_quality["tmdbId"].to_numpy(),
    catalogue_movie_ids
), "Movie ordering mismatch!"

quality_scores = movie_quality["quality_normalized"].to_numpy(
    dtype=np.float32
)

# Create the artifact dictionary
genre_artifacts = {
    "all_genres": all_genres,
    "genre_matrix": genre_matrix,
    "movie_to_idx": movie_to_idx,
    "catalogue_movie_ids": catalogue_movie_ids,
    "catalogue_titles": catalogue_titles,
    "quality_scores": quality_scores,
    "profile_method": BEST_PROFILE_METHOD,
    "alpha": BEST_ALPHA,
    "top_k": K
}

print("Genre artifacts prepared successfully!")

for key, value in genre_artifacts.items():
    if isinstance(value, np.ndarray):
        print(key, value.shape)
    else:
        print(key, type(value))


Genre artifacts prepared successfully!
all_genres <class 'list'>
genre_matrix (12262, 20)
movie_to_idx <class 'dict'>
catalogue_movie_ids (12262,)
catalogue_titles (12262,)
quality_scores (12262,)
profile_method <class 'str'>
alpha <class 'float'>
top_k <class 'int'>


In [ ]:

import pickle
from google.colab import files

genre_model_path = os.path.join(
    ARTIFACT_DIR,
    "genre_model.pkl"
)

with open(genre_model_path, "wb") as f:
    pickle.dump(
        genre_artifacts,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Genre model saved successfully!")

# Download to your local computer
files.download(genre_model_path)


Genre model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

with open(genre_model_path, "rb") as f:
    loaded_artifacts = pickle.load(f)

print("Pickle verification successful!")
print("Number of genres:", len(loaded_artifacts["all_genres"]))
print("Genre matrix shape:", loaded_artifacts["genre_matrix"].shape)
print("Movie catalogue size:", len(loaded_artifacts["catalogue_movie_ids"]))
print("Selected method:", loaded_artifacts["profile_method"])
print("Selected alpha:", loaded_artifacts["alpha"])


Pickle verification successful!
Number of genres: 20
Genre matrix shape: (12262, 20)
Movie catalogue size: 12262
Selected method: positive
Selected alpha: 0.5
